In [ ]:
from glob import glob
import pandas as pd
import os
import pyreadr
from tqdm import tqdm

In [ ]:
times = ['202002', '202003', '202103', '202203']
for target_time in times:
    print(f"processing file: {target_time}")
    info_paths = glob(f"/nas/houce/UK_mobility/processed_data/filter_day_info_{target_time}*.csv")
    for i, filepath in tqdm(enumerate(info_paths), total=len(info_paths)):
        if i == 0:
            data_counts = pd.read_csv(filepath)
            drop_columns = [col for col in data_counts.columns if col.startswith('total_records_')]
            data_counts = data_counts.drop(columns=drop_columns)
        else:
            temp_counts = pd.read_csv(filepath)
            drop_columns = [col for col in temp_counts.columns if col.startswith('total_records_')]
            temp_counts = temp_counts.drop(columns=drop_columns)
            data_counts = data_counts.merge(temp_counts, on='device_aid', how='outer')

    hours_present_cols = [col for col in data_counts.columns if col.startswith('hours_present_')]
    mask = data_counts[hours_present_cols].gt(11)
    qualified = data_counts[mask.sum(axis=1) >= 5]
    qualified_matrix = qualified[hours_present_cols]
    print(qualified_matrix.shape)
    data_counts.to_csv(f"/nas/houce/UK_mobility/processed_data/tided_filter_individuals_{target_time}.csv", index=False)

In [ ]:
from itertools import combinations
os.makedirs("/nas/houce/UK_mobility/output_data", exist_ok=True)
file_list = ['/nas/houce/UK_mobility/processed_data/tided_filter_individuals_2020.csv',
 '/nas/houce/UK_mobility/processed_data/tided_filter_individuals_202103.csv',
 '/nas/houce/UK_mobility/processed_data/tided_filter_individuals_202203.csv']
pairs = list(combinations(file_list, 2))
print(pairs)
for i in tqdm(range(len(pairs)), total=len(pairs)):
    file1, file2 = pairs[i]
    time1 = os.path.basename(file1).split('_')[-1].split('.')[0]
    time2 = os.path.basename(file2).split('_')[-1].split('.')[0]
    df1 = pd.read_csv(file1)
    df2 = pd.read_csv(file2)
    merged_df = df1.merge(df2, on='device_aid', how='outer')
    merged_df.to_csv(f"/nas/houce/UK_mobility/output_data/merged_{time1}_{time2}.csv", index=False)
    print(f"Intersection of {os.path.basename(file1)} and {os.path.basename(file2)}: {merged_df.shape}")


In [ ]:
filter_individuals = glob("/nas/houce/UK_mobility/processed_data/tided_filter_individuals_*.csv")
for i, file in tqdm(enumerate(filter_individuals), total=len(filter_individuals)):
    if i == 0:
        df = pd.read_csv(file)
    else:
        temp_df = pd.read_csv(file)
        df = df.merge(temp_df, on='device_aid', how='inner')

    print(f"{file}: {df.shape}")

In [ ]:
cols_2020 = [col for col in data_counts.columns if col.startswith('hours_present_2020')]
cols_2021 = [col for col in data_counts.columns if col.startswith('hours_present_2021')]
# filtered_5cols_2020 = data_counts[data_counts[cols_2020].gt(16).sum(axis=1) >= 2]
# filtered_5cols_2021 = data_counts[data_counts[cols_2021].gt(16).sum(axis=1) >= 2]
filtered = data_counts[(data_counts[cols_2020].gt(15).any(axis=1)) & (data_counts[cols_2021].gt(15).any(axis=1))]

In [ ]:
data = pd.read_csv(r"D:\houce\mobility\data\output\filtered_data_20210309.csv")

In [ ]:
# 按小时分组，合并同一小时内重复坐标的记录
def merge_duplicates(df):
    # 按小时分组
    merged = []
    for hour, group in df.groupby('hours'):
        # 保留原始顺序
        group = group.reset_index(drop=True)
        i = 0
        while i < len(group):
            current_row = group.iloc[i].copy()
            current_lat, current_lon = current_row['latitude'], current_row['longitude']
            
            # 计算连续重复的记录数
            count = 1
            j = i + 1
            while j < len(group) and group.iloc[j]['latitude'] == current_lat and group.iloc[j]['longitude'] == current_lon:
                count += 1
                j += 1
            
            # 调整duration（乘以重复条数）
            current_row['duration'] = current_row['duration'] * count
            merged.append(current_row)
            
            # 跳过已处理的连续记录
            i = j
            
    return pd.DataFrame(merged)

id = "REDACTED_DEVICE_ID" # 一个移动很多的例子
# id = "REDACTED_DEVICE_ID"

df = data[data['device_aid'] == id]
df['duration'] = 60 / df.groupby('hours')['hours'].transform('count')
df['duration'] = df['duration'].round(2)
print(df.shape)
df_merged = merge_duplicates(df)
print(df_merged.shape)
df_merged['status'] = 'moving'  # Default all to moving

# Get previous row's coordinates (shift)
df_merged['prev_latitude'] = df_merged['latitude'].shift()
df_merged['prev_longitude'] = df_merged['longitude'].shift()
df_merged.loc[(df_merged['duration'] > 5), 'status'] = 'poi'
# If coordinates are the same as previous hour, mark as 'poi'
df_merged.loc[(df_merged['latitude'] == df_merged['prev_latitude']) & 
    (df_merged['longitude'] == df_merged['prev_longitude']), 'status'] = 'poi'

# Drop the temporary columns used for calculation
df_merged = df_merged.drop(columns=['prev_latitude', 'prev_longitude'])
df_merged = df_merged.drop(df_merged.index[0])

In [ ]:
location_poi_label = pd.DataFrame(df_merged[df_merged['status'] == 'poi'].value_counts(['latitude','longitude'])).reset_index()
location_poi_label['label'] = 'amenity'
location_poi_label.loc[0, 'label'] = 'home'
location_poi_label.loc[1, 'label'] = 'workplace'
df_merged = df_merged.merge(location_poi_label, on=['latitude', 'longitude'], how='left')